In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# US state to region mapping
state_to_region = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Michigan': 'Midwest', 'Ohio': 'Midwest',
    'Wisconsin': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest', 'Minnesota': 'Midwest',
    'Missouri': 'Midwest', 'Nebraska': 'Midwest', 'North Dakota': 'Midwest',
    'South Dakota': 'Midwest', 'Delaware': 'South', 'District of Columbia': 'South',
    'Florida': 'South', 'Georgia': 'South', 'Maryland': 'South', 'North Carolina': 'South',
    'South Carolina': 'South', 'Virginia': 'South', 'West Virginia': 'South',
    'Alabama': 'South', 'Kentucky': 'South', 'Mississippi': 'South', 'Tennessee': 'South',
    'Arkansas': 'South', 'Louisiana': 'South', 'Oklahoma': 'South', 'Texas': 'South',
    'Arizona': 'West', 'Colorado': 'West', 'Idaho': 'West', 'Montana': 'West',
    'Nevada': 'West', 'New Mexico': 'West', 'Utah': 'West', 'Wyoming': 'West',
    'Alaska': 'West', 'California': 'West', 'Hawaii': 'West', 'Oregon': 'West',
    'Washington': 'West'
}

# Map regions
df['Region'] = df['Ecoregion'].map(state_to_region)

# Filter forest loss and recovery
loss_df = df[df['ForestChangeType'].isin([10, 20, 30, 40, 50])]
gain_df = df[df['ForestChangeType'].isin([1, 2, 3, 4, 5])]

# Weighted mean gap year by disturbance and region
loss_stats = loss_df.groupby(['Region', 'DisturbanceCategory']) \
    .apply(lambda g: (g['GapYears'] * g['PixelCount']).sum() / g['PixelCount'].sum()) \
    .reset_index(name='WeightedMeanGapYears_Loss')

gain_stats = gain_df.groupby(['Region', 'DisturbanceCategory']) \
    .apply(lambda g: (g['GapYears'] * g['PixelCount']).sum() / g['PixelCount'].sum()) \
    .reset_index(name='WeightedMeanGapYears_Gain')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# === Define your disturbance color map ===
disturbance_colors = {
    'No Disturbance Detected': 'black',
    'Logging': '#1b9e77',
    'Construction': 'purple',
    'Stress': '#e7298a',
    'Natural Hazard': '#66a61e',
    'Water Dynamic': '#1f78b4',
    'Fire': 'red',
    'Agriculture Activity': 'gold',
    'Other': 'gray'
}

# === Step 1: Add Region ===
df['Region'] = df['Ecoregion'].map(state_to_region)

# === Step 2: Filter and aggregate ===
loss_df = df[df['ForestChangeType'].isin([10, 20, 30, 40, 50])].copy()
gain_df = df[df['ForestChangeType'].isin([1, 2, 3, 4, 5])].copy()

# Function to calculate weighted mean gap years
def weighted_gap(df, label):
    stats = df.groupby(['Region', 'DisturbanceCategory']).apply(
        lambda g: (g['GapYears'] * g['PixelCount']).sum() / g['PixelCount'].sum()
    ).reset_index(name=f'WeightedMeanGapYears_{label}')
    return stats

loss_stats = weighted_gap(loss_df, "Loss")
gain_stats = weighted_gap(gain_df, "Gain")

# Add CONUS as a new row
def add_conus(df, label):
    all_region = df.groupby(['DisturbanceCategory']).apply(
        lambda g: (g['GapYears'] * g['PixelCount']).sum() / g['PixelCount'].sum()
    ).reset_index(name=f'WeightedMeanGapYears_{label}')
    all_region['Region'] = 'CONUS'
    return all_region[['Region', 'DisturbanceCategory', f'WeightedMeanGapYears_{label}']]

loss_conus = add_conus(loss_df, "Loss")
gain_conus = add_conus(gain_df, "Gain")

# Combine regional and CONUS data
loss_all = pd.concat([loss_stats, loss_conus], ignore_index=True)
gain_all = pd.concat([gain_stats, gain_conus], ignore_index=True)

In [ ]:
import matplotlib.patches as mpatches

# === Step 3: Merge and melt for plotting ===
summary_df = pd.merge(loss_all, gain_all, how='outer', on=['Region', 'DisturbanceCategory'])

df_melted = summary_df.melt(
    id_vars=["Region", "DisturbanceCategory"],
    value_vars=["WeightedMeanGapYears_Loss", "WeightedMeanGapYears_Gain"],
    var_name="ChangeType",
    value_name="MeanGapYears"
)
# === Fix disturbance naming to match color mapping ===
disturbance_rename_dict = {
    'Forest Management': 'Logging',
    'No Disturbance': 'No Disturbance Detected'
}

df_melted["DisturbanceCategory"] = df_melted["DisturbanceCategory"].replace(disturbance_rename_dict)

df_melted["ChangeType"] = df_melted["ChangeType"].str.replace("WeightedMeanGapYears_", "")
df_melted["Color"] = df_melted["DisturbanceCategory"].map(disturbance_colors)
df_melted["Color"] = df_melted["Color"].fillna("lightgray")
df_melted["IsCONUS"] = df_melted["Region"] == "CONUS"
df_melted = df_melted.sort_values(by=["DisturbanceCategory", "Region"])
# Assign hatch patterns to each region
region_hatches = {
    'CONUS': '///',
    'Northeast': '\\\\',
    'Midwest': 'xx',
    'South': '--',
    'West': '++'
}
df_melted["Hatch"] = df_melted["Region"].map(region_hatches).fillna("")

# === ompute average gap years per disturbance category ===
disturbance_order = (
    df_melted
    .groupby("DisturbanceCategory")["MeanGapYears"]
    .mean()
    .sort_values(ascending=False)  # or ascending=True if you prefer short → long
    .index
    .tolist()
)

df_melted["DisturbanceCategory"] = pd.Categorical(
    df_melted["DisturbanceCategory"],
    categories=disturbance_order,
    ordered=True
)

df_melted = df_melted.sort_values(by=["DisturbanceCategory", "Region"])

# === Step 4: Plotting ===
def plot_loss_gain_bars(df_melted):
    df_melted = df_melted.sort_values(by=["DisturbanceCategory", "Region"])

    fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)

    for i, change_type in enumerate(["Loss", "Gain"]):
        ax = axes[i]
        subset = df_melted[df_melted["ChangeType"] == change_type]

        x = list(range(len(subset)))
        bars = ax.bar(
            x=x,
            height=subset["MeanGapYears"],
            color=subset["Color"],
            edgecolor='darkgray',
            hatch=subset["Hatch"],
            linewidth=1
        )

        # === Add Region name as x tick label ===
        region_labels = subset["Region"].tolist()
        ax.set_xticks(x)
        ax.set_xticklabels(region_labels, rotation=90, fontsize=8)
        
        panel_label = "(a)" if change_type == "Loss" else "(b)"
        ax.set_title(
            f"{panel_label} Mean Gap Years for Forest {change_type} After Disturbances",
            loc='left',            # left-aligned
            fontweight='bold'      # bold
            )
        ax.set_xlabel("Region")
        ax.set_ylabel("Mean Gap Years")

    # === Construct legends ===

    # Color legend (Disturbance type)
    disturbance_legend = [
        mpatches.Patch(color=color, label=dist)
        for dist, color in disturbance_colors.items()
    ]

    # Hatch legend (Region)
    region_legend = [
        mpatches.Patch(facecolor='white', edgecolor='black', hatch=hatch, label=region)
        for region, hatch in region_hatches.items()
    ]

    # Add legends below plot
    fig.legend(handles=disturbance_legend, loc='upper center', bbox_to_anchor=(0.76, 0.88), ncol=2)

    
    plt.subplots_adjust(wspace=0.1, bottom=0.25)  # Reduce horizontal gap and make space for legend
    plt.savefig(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Figure_S5.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_loss_gain_bars(df_melted)